In [1]:
import numpy as np

In [16]:
def softmax(Z):
    Z -= np.max(Z, axis=-1, keepdims=True)
    exp = np.exp(Z)
    return exp/np.sum(exp, axis=-1, keepdims=True)

def multihead_attention(X, W_Q, W_K, W_V, W_O=None, h=None, mask=None):

    N, seq_len, d_model = X.shape
    d_k = d_model//h

    # step 1: Project to Q, K, V
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    # step 2: Reshape and transpose into heads

    reshaped_Q = Q.reshape(N, seq_len, h, d_k)
    multi_head_Q = reshaped_Q.transpose(0, 2, 1, 3)

    reshaped_K = K.reshape(N, seq_len, h, d_k)
    multi_head_K = reshaped_K.transpose(0, 2, 1, 3)

    reshaped_V = V.reshape(N, seq_len, h, d_k)
    multi_head_V = reshaped_V.transpose(0, 2, 1, 3)

    scores = multi_head_Q @ multi_head_K.transpose(0, 1, 3, 2)
    scaled_scores = scores / np.sqrt(d_k)

    if mask is not None:
        scaled_scores += mask

    weights = softmax(scaled_scores)
    multi_head_outputs = weights @ multi_head_V

    concat_outputs = multi_head_outputs.transpose(0, 2, 1, 3)
    concat_outputs = concat_outputs.reshape(N, seq_len, d_model)

    projected_outputs = concat_outputs @ W_O

    return projected_outputs

    








In [17]:
N, seq_len, d_model, h = 2, 4, 8, 2
d_k = d_model // h
rng = np.random.default_rng(42)

X   = rng.standard_normal((N, seq_len, d_model))
W_Q = rng.standard_normal((d_model, d_model))
W_K = rng.standard_normal((d_model, d_model))
W_V = rng.standard_normal((d_model, d_model))
W_O = rng.standard_normal((d_model, d_model))

mask = np.triu(np.ones((seq_len, seq_len)), k=1)
mask[mask == 1] = -np.inf

out = multihead_attention(X, W_Q, W_K, W_V, W_O, h=h, mask=mask)
print(out.shape)   # (2, 4, 8) ✅

(2, 4, 8)


In [7]:
rng = np.random.default_rng(42)
X = rng.standard_normal((2, 8, 4))
w_q = rng.standard_normal((4, 4))
w_k = rng.standard_normal((4, 4))
w_v = rng.standard_normal((4, 4))

multihead_attention(X, w_q, w_k, w_v, None, 2)


(array([[[[-1.76742123,  0.96291162],
          [-2.48729436, -1.28531557],
          [-1.88471869,  0.71037489],
          [ 0.951391  , -0.06706552],
          [-1.34679495,  0.90480157],
          [-1.77374545,  0.56690523],
          [-1.25233007,  0.0501326 ],
          [-1.13232046,  1.24533256]],
 
         [[ 1.91572283,  1.26492932],
          [ 2.16191691,  2.56390848],
          [ 1.70590352,  1.341116  ],
          [-1.9320125 , -0.71938995],
          [ 1.18852154,  1.53819828],
          [ 1.01056784,  1.74911024],
          [ 0.90301343,  0.85160943],
          [-0.64274627,  0.92542778]]],
 
 
        [[[-2.11123811,  0.2345196 ],
          [-0.32177281, -0.23642925],
          [ 1.50734827,  0.19169236],
          [-0.62614938,  0.51942524],
          [ 0.00529153,  0.76950909],
          [-0.60519068, -1.43001437],
          [ 1.46932602, -0.78121877],
          [-1.88433371, -1.09718118]],
 
         [[ 2.01075617,  1.23515567],
          [ 1.40108972,  0.23427157],


In [5]:
arr = np.array([1, 2, 3])

arr1 = arr.reshape(1, 3)

arr is arr1

False

In [ ]:
arr = np.array([
    [
        [[1, 2],
         [2, 3]],
        [ [2, 3],
         [2, 1]]
    ]
])

arr1 = np.array([[1, 2], 
                 [2, 1]])

In [15]:
print(arr1.shape)
print(arr.shape)

(2, 2)
(1, 2, 2, 2)


In [14]:
arr @ arr1

array([[[[5, 4],
         [8, 7]],

        [[8, 7],
         [4, 5]]]])

In [13]:
arr1 @ arr

array([[[[5, 8],
         [4, 7]],

        [[6, 5],
         [6, 7]]]])

In [ ]:
def softmax(Z):
    Z = Z - np.max(Z, axis=-1, keepdims=True)
    exp = np.exp(Z)
    return exp / np.sum(exp, axis=-1, keepdims=True)


def multihead_attention(X, W_Q, W_K, W_V, W_O, h, mask=None):

    N, seq_len, d_model = X.shape
    d_k = d_model // h

    # step 1: Projection
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    # step 2: Multi-headed Transformation
    Q_transformed = Q.resize(N, seq_len, h, d_k)
    K_transformed = K.resize(N, seq_len, h, d_k)
    V_transformed = V.resize(N, seq_len, h, d_k)

    multihead_Q = Q_transformed.transpose(0, 2, 1, 3)
    multihead_K = K_transformed.transpose(0, 2, 1, 3)
    multihead_V = V_transformed.transpose(0, 2, 1, 3)

    # Step 3: Q @ K
    scores = multihead_Q @ multihead_K.transpose(0, 2, 3, 1)
    scaled_scores = scores / np.sqrt(d_k)

    if mask is not None:
        scaled_scores += mask

    weights = softmax(scaled_scores)

    multihead_output = weights @ multihead_V
    concat_output = multihead_output.transpose(0, 2, 1, 3).resize(N, seq_len, d_model)
    final_proj_output = concat_output @ W_O
    return final_proj_output
